In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix

print("=" * 60)
print("   KNN Example 1: Pumpkin Seed Classification")
print("=" * 60)
print()


# STEP 2: CREATE THE DATASET

np.random.seed(42)
n1 = 125

cercevelik = pd.DataFrame({
    'Area':        np.random.normal(55000, 5000, n1),
    'Perimeter':   np.random.normal(850, 40, n1),
    'Major_Axis':  np.random.normal(310, 20, n1),
    'Minor_Axis':  np.random.normal(220, 15, n1),
    'Roundness':   np.random.normal(0.88, 0.04, n1),
    'Compactness': np.random.normal(0.72, 0.03, n1),
    'Class':       'Cercevelik'
})

n2 = 125

urgup = pd.DataFrame({
    'Area':        np.random.normal(72000, 6000, n2),
    'Perimeter':   np.random.normal(1050, 50, n2),
    'Major_Axis':  np.random.normal(420, 25, n2),
    'Minor_Axis':  np.random.normal(210, 15, n2),
    'Roundness':   np.random.normal(0.75, 0.05, n2),
    'Compactness': np.random.normal(0.55, 0.04, n2),
    'Class':       'Urgup_Sivrisi'
})

df = pd.concat([cercevelik, urgup], ignore_index=True)

print("STEP 2: Dataset Overview")
print("-" * 40)
print(f"Total number of seeds: {df.shape[0]}")
print(f"Number of features:    {df.shape[1] - 1}")
print()
print("First 10 rows of the dataset:")
print(df.head(10).to_string(index=False))
print()


# STEP 3: EXPLORE THE DATA
print("STEP 3: Exploring the Data")
print("-" * 40)
print("Average values per seed variety:")
print(df.groupby('Class').mean().round(1))
print()

plt.figure(figsize=(8, 5))

colors = {'Cercevelik': '#2ecc71', 'Urgup_Sivrisi': '#e74c3c'}

for seed_type, color in colors.items():
    subset = df[df['Class'] == seed_type]
    plt.scatter(
        subset['Roundness'],
        subset['Area'],
        c=color,
        label=seed_type,
        alpha=0.6,
        edgecolors='w',
        s=60
    )

plt.xlabel('Roundness', fontsize=12)
plt.ylabel('Area (pixels)', fontsize=12)
plt.title('Pumpkin Seeds: Roundness vs Area', fontsize=14)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('pumpkin_scatter.png', dpi=150)
plt.show()

print("   -> Scatter plot saved as 'pumpkin_scatter.png'")
print("   -> Look for two distinct clusters. If they're well-separated,")
print("      KNN should perform well!")
print()


# STEP 4: PREPARE THE DATA FOR MACHINE LEARNING
print("STEP 4: Preparing the Data")
print("-" * 40)

X = df.drop('Class', axis=1)
y = df['Class']

print(f"Features (X) shape: {X.shape}")
print(f"Labels (y) shape:   {y.shape}")
print()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")
print()

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features have been scaled using StandardScaler.")
print("   -> Training data: fit_transform (learn + scale)")
print("   -> Testing data:  transform only (scale using same parameters)")
print()


# STEP 5: BUILD AND TRAIN THE KNN MODEL

print("STEP 5: Training the KNN Model")
print("-" * 40)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

print("Model trained with K=5 neighbors.")
print()

y_pred = knn.predict(X_test_scaled)

print("Predictions made on the test set.")
print()


# STEP 6: EVALUATE THE MODEL
print("STEP 6: Evaluating the Model")
print("-" * 40)

accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy:.2%}")
print()

print("Detailed Classification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Cercevelik', 'Urgup'],
    yticklabels=['Cercevelik', 'Urgup']
)
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title('Confusion Matrix — Pumpkin Seeds')
plt.tight_layout()
plt.savefig('pumpkin_confusion.png', dpi=150)
plt.show()

print("   -> Confusion matrix saved as 'pumpkin_confusion.png'")
print()


# STEP 7: FIND THE BEST VALUE OF K
print("STEP 7: Finding the Best K")
print("-" * 40)

k_values = range(1, 26)
accuracies = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, pred)
    accuracies.append(acc)

plt.figure(figsize=(8, 4))
plt.plot(
    k_values,
    accuracies,
    marker='o',
    color='#3498db',
    linewidth=2,
    markersize=5
)
plt.xlabel('K (Number of Neighbors)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Finding the Best K — Pumpkin Seeds', fontsize=14)
plt.xticks(k_values)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('pumpkin_best_k.png', dpi=150)
plt.show()

best_k = list(k_values)[np.argmax(accuracies)]
best_accuracy = max(accuracies)

print(f"   -> Best K = {best_k}")
print(f"   -> Best Accuracy = {best_accuracy:.2%}")
print(f"   -> Chart saved as 'pumpkin_best_k.png'")
print()


# STEP 8: MAKE A PREDICTION ON NEW DATA

print("STEP 8: Predicting a New Seed")
print("-" * 40)

final_model = KNeighborsClassifier(n_neighbors=best_k)
final_model.fit(X_train_scaled, y_train)
new_seed = pd.DataFrame({
    'Area':        [60000],
    'Perimeter':   [920],
    'Major_Axis':  [350],
    'Minor_Axis':  [215],
    'Roundness':   [0.82],
    'Compactness': [0.65],
})

print("New seed measurements:")
print(new_seed.to_string(index=False))
print()

new_seed_scaled = scaler.transform(new_seed)

prediction = final_model.predict(new_seed_scaled)
print(f"Predicted variety: {prediction[0]}")
probabilities = final_model.predict_proba(new_seed_scaled)

print("Confidence scores:")
for class_name, prob in zip(final_model.classes_, probabilities[0]):
    bar = '█' * int(prob * 30)
    print(f"   {class_name:15s} {prob:6.1%}  {bar}")

print()
print("=" * 60)
print("   DONE! You've completed the Pumpkin Seed KNN example.")
print("=" * 60)
